# V1DD 1196 CCM ETL

Convert local V1DD release files to CCM tables.

In [ ]:
import gc
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.feather as feather
from deltalake import DeltaTable, write_deltalake

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import (
    cell_cell_connectivity_to_arrow,
    derive_cell_cell_connectivity,
    read_cell_cell_connectivity,
    read_embedding_locations,
    write_models,
)
from connects_common_connectivity.io.arrow_utils import (
    attach_linkml_metadata,
    build_arrow_schema,
    build_cell_feature_matrix_schema,
)
from connects_common_connectivity.io.path_spec import (
    MODEL_TABLE_PATHS,
    WIDE_PAYLOAD_PATHS,
)
from connects_common_connectivity.models import (
    AlgorithmRun,
    AnatomicalOrientation,
    CellFeatureDefinition,
    CellFeatureMatrix,
    CellFeatureSet,
    Cluster,
    ClusterHierarchy,
    ClusterMembership,
    DataItem,
    DataItemDataSetAssociation,
    DataSet,
    Default2DView,
    EmbeddingLocation,
    EmbeddingMethod,
    EmbeddingSpace,
    HierarchyCategory,
    LocationType,
    Modality,
    ReferenceSpace,
    SignedAxis,
    SpatialLocation,
    SynapseConnectivityLong,
    SynapseFeatureMatrix,
    Unit,
)

In [ ]:
REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = REPO_ROOT / "data/EM v1dd 1196"
COLORS_PATH = REPO_ROOT / "data/EM cell types/em_cell_type_colors_csm_202610.rtf"
OUTPUT_ROOT = Path(get_settings().output_root)

PROJECT_ID = "v1dd_1196"
DATASET_EM = "v1dd_1196_all"
DATASET_PROOFREAD_AXON = "v1dd_1196_proofread_axons"
DATASET_PROOFREAD_DEND = "v1dd_1196_proofread_dendrites"
CAVE_REFERENCE_SPACE_ID = "v1dd_cave_nm"
STREAMLINE_REFERENCE_SPACE_ID = "v1dd_streamline_nm"
VOLUME_FEATURE_SET_ID = "v1dd_soma_volume"
SOMA_FEATURE_SET_ID = "v1dd_soma_features"
HIERARCHY_ID = "EM_csm_cell_types"
RUN_ID = "v1dd_cell_type_annotation_r84"
ROOT_ID = "neuron"
SYNAPSE_TABLE_ID = "v1dd_1196_synapses"
SYNAPSE_FEATURE_MATRIX_ID = "v1dd_1196_synapse_features"
CONNECTOME_ID = "v1dd_1196_em_proofread_axon_to_dendrite"
QC_EMBEDDING_ID = "v1dd_r84_qc_nonnull_umap"
AD_REF_EMBEDDING_ID = "v1dd_r84_ad_ref_umap"
PUBLICATION = "https://github.com/AllenInstitute/v1dd_physiology"

SOMA_PATH = DATA_ROOT / "soma_and_cell_type_1196.feather"
R84_PATH = DATA_ROOT / "features_axon_dendrite_tables_v1dd_cell_type_axon_dendrite--r84.parquet"
AXON_PATH = DATA_ROOT / "proofread_axon_list_1196.npy"
DENDRITE_PATH = DATA_ROOT / "proofread_dendrite_list_1196.npy"
SYNAPSE_PATH = DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather"
SYNAPSE_LABEL_PATH = DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather"
SOMA_FEATURES_PATH = DATA_ROOT / "soma_features.parquet"

required_paths = [
    SOMA_PATH, R84_PATH, AXON_PATH, DENDRITE_PATH, SYNAPSE_PATH,
    SYNAPSE_LABEL_PATH, SOMA_FEATURES_PATH, COLORS_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
assert not missing_paths, f"Missing local inputs: {missing_paths}"

## dataset dataitem

In [ ]:
soma_source = pd.read_feather(SOMA_PATH)
r84_source = pd.read_parquet(R84_PATH)

# Cell-type annotations use nucleus ids; CCM DataItems use root ids so every
# downstream table refers to the reconstructed cell rather than the nucleus row.
cell_type_df = r84_source.rename(
    columns={"root_id": "r84_root_id", "cell_id": "id"}
)

soma_columns = {
    "id", "pt_root_id", "volume",
    "pt_position_x", "pt_position_y", "pt_position_z",
    "pt_position_trform_x", "pt_position_trform_y", "pt_position_trform_z",
}
cell_type_columns = [
    "class", "subclass", "subclass_confidence",
    "cluster", "cluster_confidence",
]
assert soma_columns <= set(soma_source.columns)
assert {"id", *cell_type_columns} <= set(cell_type_df.columns)
assert cell_type_df["id"].is_unique
assert set(cell_type_df["class"].dropna().unique()) == {"exc", "inh"}
assert cell_type_df["subclass"].notna().all()

soma_ct_df = soma_source[list(soma_columns)].merge(
    cell_type_df[["id"] + cell_type_columns],
    how="left",
    on="id",
    validate="many_to_one",
)

# DataItem identity must be unambiguous. Remove both one-nucleus-to-many-root and
# many-nuclei-to-one-root mappings instead of choosing an arbitrary record.
id_root_pairs = soma_ct_df[["id", "pt_root_id"]].drop_duplicates()
root_ids_per_id = id_root_pairs.groupby("id")["pt_root_id"].nunique()
ids_per_root_id = id_root_pairs.groupby("pt_root_id")["id"].nunique()
ambiguous_ids = root_ids_per_id.index[root_ids_per_id > 1]
ambiguous_root_ids = ids_per_root_id.index[ids_per_root_id > 1]
ambiguous_mask = (
    soma_ct_df["id"].isin(ambiguous_ids)
    | soma_ct_df["pt_root_id"].isin(ambiguous_root_ids)
)
soma_ct_unique_df = soma_ct_df.loc[~ambiguous_mask].reset_index(drop=True)
assert soma_ct_unique_df["id"].is_unique
assert soma_ct_unique_df["pt_root_id"].is_unique

# Dataset associations may only reference registered DataItems, so trim cohort
# lists to roots retained by the identity cleanup.
unique_root_ids = np.array(sorted(set(soma_ct_unique_df["pt_root_id"])))
registered_ids = set(soma_ct_unique_df["pt_root_id"].astype(str))
axon_source_ids = np.load(AXON_PATH, allow_pickle=True)
dendrite_source_ids = np.load(DENDRITE_PATH, allow_pickle=True)
axon_proof_root_ids = axon_source_ids[np.isin(axon_source_ids, unique_root_ids)]
dendrite_proof_root_ids = dendrite_source_ids[
    np.isin(dendrite_source_ids, unique_root_ids)
]
print(f"clean somas: {len(soma_ct_unique_df):,}")
print(f"proofread axons: {len(axon_proof_root_ids):,} / {len(axon_source_ids):,}")
print(
    f"proofread dendrites: {len(dendrite_proof_root_ids):,} / "
    f"{len(dendrite_source_ids):,}"
)

In [ ]:
datasets = [
    DataSet(
        id=DATASET_EM, name="V1DD release 1196 - EM somas",
        publication=PUBLICATION, modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_AXON, name="V1DD release 1196 - proofread axons cohort",
        publication=PUBLICATION, modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
    DataSet(
        id=DATASET_PROOFREAD_DEND, name="V1DD release 1196 - proofread dendrites cohort",
        publication=PUBLICATION, modality=Modality.ELECTRON_MICROSCOPY,
        project_id=PROJECT_ID,
    ),
]
print(write_models(datasets, output_root=OUTPUT_ROOT))

dataitems = [
    DataItem(id=str(row.pt_root_id), name=str(row.id), project_id=PROJECT_ID)
    for row in soma_ct_unique_df.itertuples()
]
print(write_models(dataitems, output_root=OUTPUT_ROOT))

associations = [
    DataItemDataSetAssociation(
        dataitem_id=str(root_id), dataset_id=DATASET_EM, project_id=PROJECT_ID
    )
    for root_id in unique_root_ids
]
associations.extend(
    DataItemDataSetAssociation(
        dataitem_id=str(root_id), dataset_id=DATASET_PROOFREAD_AXON, project_id=PROJECT_ID
    )
    for root_id in axon_proof_root_ids
)
associations.extend(
    DataItemDataSetAssociation(
        dataitem_id=str(root_id), dataset_id=DATASET_PROOFREAD_DEND, project_id=PROJECT_ID
    )
    for root_id in dendrite_proof_root_ids
)
assert {row.dataitem_id for row in associations} <= registered_ids
print(write_models(associations, output_root=OUTPUT_ROOT))

## spatial location

In [ ]:
default_view = Default2DView(
    left_to_right=SignedAxis.PLUS_X,
    bottom_to_top=SignedAxis.MINUS_Y,
)
cortical_orientation = AnatomicalOrientation(
    pia_to_white_matter=SignedAxis.PLUS_Y,
)
reference_spaces = [
    ReferenceSpace(
        id=CAVE_REFERENCE_SPACE_ID,
        name="V1DD CAVE nanometer coordinates",
        description="Soma nucleus positions from the local release table.",
        unit=Unit.NANOMETERS_LENGTH,
        default_2d_view=default_view,
        anatomical_orientation=cortical_orientation,
        project_id=PROJECT_ID,
    ),
    ReferenceSpace(
        id=STREAMLINE_REFERENCE_SPACE_ID,
        name="V1DD streamline-transformed nanometer coordinates",
        description="Precomputed V1DD streamline-transformed soma positions.",
        unit=Unit.NANOMETERS_LENGTH,
        default_2d_view=default_view,
        anatomical_orientation=cortical_orientation,
        project_id=PROJECT_ID,
    ),
]
print(write_models(reference_spaces, output_root=OUTPUT_ROOT))

spatial_locations = []
for row in soma_ct_unique_df.itertuples():
    common = {
        "dataitem_id": str(row.pt_root_id),
        "location_type": LocationType.SOMA,
        "project_id": PROJECT_ID,
    }
    spatial_locations.extend([
        SpatialLocation(
            reference_space=CAVE_REFERENCE_SPACE_ID,
            description="Soma nucleus position from the local release table.",
            x=float(row.pt_position_x), y=float(row.pt_position_y), z=float(row.pt_position_z),
            **common,
        ),
        SpatialLocation(
            reference_space=STREAMLINE_REFERENCE_SPACE_ID,
            description="Streamline-transformed soma nucleus position.",
            x=float(row.pt_position_trform_x),
            y=float(row.pt_position_trform_y),
            z=float(row.pt_position_trform_z),
            **common,
        ),
    ])
print(write_models(spatial_locations, output_root=OUTPUT_ROOT))

## features

In [ ]:
volume_definition = CellFeatureDefinition(
    id="soma_volume",
    description="Nucleus volume from the local V1DD release table.",
    unit=Unit.MICRONS_CUBED.value,
    data_type="<f4",
    project_id=PROJECT_ID,
    feature_set_id=VOLUME_FEATURE_SET_ID,
)
volume_feature_set = CellFeatureSet(
    id=VOLUME_FEATURE_SET_ID,
    description="V1DD release 1196 soma nucleus volume.",
    feature_definition_ids=[volume_definition.id],
    extraction_method="Loaded from soma_and_cell_type_1196.feather.",
    project_id=PROJECT_ID,
)
print(write_models([volume_definition], output_root=OUTPUT_ROOT))
print(write_models([volume_feature_set], output_root=OUTPUT_ROOT))

# Wide feature payloads use the DataItem id as their row key and carry project
# and feature-set scope columns alongside the measured values.
volume_wide = pd.DataFrame({
    "id": soma_ct_unique_df["pt_root_id"].astype(str),
    "soma_volume": soma_ct_unique_df["volume"].astype("float32"),
    "project_id": PROJECT_ID,
    "feature_set_id": VOLUME_FEATURE_SET_ID,
})
volume_schema = build_cell_feature_matrix_schema(
    volume_feature_set, [volume_definition], cell_index_column="id"
)
volume_table = pa.Table.from_pandas(
    volume_wide, schema=volume_schema, preserve_index=False
)
volume_path = (
    OUTPUT_ROOT / WIDE_PAYLOAD_PATHS["cell_features"] / VOLUME_FEATURE_SET_ID
)
write_deltalake(
    volume_path, volume_table, mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)

# CellFeatureMatrix is the CCM metadata pointer to the dataframe-backed payload.
volume_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{VOLUME_FEATURE_SET_ID}",
    feature_set_id=VOLUME_FEATURE_SET_ID,
    parquet_path=f"file://{volume_path.resolve()}/",
    cell_index_column="id",
    project_id=PROJECT_ID,
)
print(write_models([volume_matrix], output_root=OUTPUT_ROOT))

## cell types

In [ ]:
CLASS_CATEGORY = "Class"
SUBCLASS_CATEGORY = "Sub-class"
CLUSTER_CATEGORY = "Cluster"

# Only cells with a complete class/subclass path can enter the hierarchy. A
# source cluster is optional, so some memberships stop at the subclass level.
ct_df = soma_ct_unique_df.dropna(subset=["class", "subclass"]).copy()
subclass_to_class_values = ct_df.groupby("subclass")["class"].unique()
assert all(len(values) == 1 for values in subclass_to_class_values)
subclass_to_class = {
    key: values[0] for key, values in subclass_to_class_values.items()
}
class_ids = sorted(ct_df["class"].unique())
subclass_ids = sorted(ct_df["subclass"].unique())
cluster_pairs = sorted(set(
    ct_df.dropna(subset=["cluster"])[["subclass", "cluster"]]
    .itertuples(index=False, name=None)
))

# Cluster labels can repeat under different subclasses; prefix leaf ids with
# their parent so Cluster ids remain unique within the hierarchy.
def cluster_node_id(subclass, cluster):
    return f"{subclass}:{cluster}"

# CCM stores the explicit tree: root -> class -> subclass -> source cluster.
clusters = [Cluster(
    id=ROOT_ID, hierarchy_id=HIERARCHY_ID, children=class_ids, level=0
)]
for class_id in class_ids:
    children = sorted(
        subclass for subclass in subclass_ids
        if subclass_to_class[subclass] == class_id
    )
    clusters.append(Cluster(
        id=class_id, hierarchy_id=HIERARCHY_ID, parent=ROOT_ID,
        children=children, level=1, hierarchy_category=CLASS_CATEGORY,
    ))
for subclass in subclass_ids:
    children = [
        cluster_node_id(parent, cluster)
        for parent, cluster in cluster_pairs if parent == subclass
    ]
    clusters.append(Cluster(
        id=subclass, hierarchy_id=HIERARCHY_ID,
        parent=subclass_to_class[subclass], children=children or None,
        level=2, hierarchy_category=SUBCLASS_CATEGORY,
    ))
for subclass, source_cluster in cluster_pairs:
    clusters.append(Cluster(
        id=cluster_node_id(subclass, source_cluster),
        hierarchy_id=HIERARCHY_ID,
        parent=subclass,
        level=3,
        hierarchy_category=CLUSTER_CATEGORY,
    ))
assert len(clusters) == 149
print(write_models(clusters, output_root=OUTPUT_ROOT))

In [ ]:
run = AlgorithmRun(
    id=RUN_ID,
    algorithm_name="V1DD CSM cell-type annotation",
    algorithm_version="r84",
    input_dataset=DATASET_EM,
    json_object=json.dumps({
        "source": str(R84_PATH),
        "assignment_columns": ["class", "subclass", "cluster"],
        "confidence_columns": ["subclass_confidence", "cluster_confidence"],
    }),
)
categories = [
    HierarchyCategory(
        id=CLASS_CATEGORY, hierarchy_id=HIERARCHY_ID,
        description="Cell class from the r84 parquet class column.", level=1,
    ),
    HierarchyCategory(
        id=SUBCLASS_CATEGORY, hierarchy_id=HIERARCHY_ID,
        description="Cell sub-class from the r84 parquet subclass column.", level=2,
    ),
    HierarchyCategory(
        id=CLUSTER_CATEGORY, hierarchy_id=HIERARCHY_ID,
        description="Cell cluster from the r84 parquet cluster column.", level=3,
    ),
]
hierarchy = ClusterHierarchy(
    id=HIERARCHY_ID,
    run=RUN_ID,
    root=ROOT_ID,
    clusters=[cluster.id for cluster in clusters],
)
print(write_models([run], output_root=OUTPUT_ROOT))
print(write_models(categories, output_root=OUTPUT_ROOT))
print(write_models([hierarchy], output_root=OUTPUT_ROOT))

# Store one membership for every assigned ancestor. This denormalization lets
# consumers filter cells at any hierarchy level without recursively walking it.
memberships = []
for row in ct_df.to_dict("records"):
    assigned = [ROOT_ID, row["class"], row["subclass"]]
    if pd.notna(row["cluster"]):
        assigned.append(cluster_node_id(row["subclass"], row["cluster"]))
    memberships.extend(
        ClusterMembership(
            item=str(row["pt_root_id"]),
            cluster=cluster_id,
            hierarchy_id=HIERARCHY_ID,
            probability=1.0,
            project_id=PROJECT_ID,
        )
        for cluster_id in assigned
    )
print(write_models(memberships, output_root=OUTPUT_ROOT))
print(f"annotated cells: {len(ct_df):,} | memberships: {len(memberships):,}")

## synapses connectivity

In [ ]:
# Release completed model batches before loading the multi-million-row synapse
# table; CCM bulk tables are written column-wise instead of as Pydantic objects.
del spatial_locations, associations, dataitems, memberships, volume_wide, volume_table
gc.collect()

synapse_source = feather.read_table(SYNAPSE_PATH, memory_map=False)
synapse_count = synapse_source.num_rows
assert pc.count_distinct(synapse_source["id"]).as_py() == synapse_count

# SynapseConnectivityLong contains only connection identity and table scope.
# Per-synapse measurements are stored separately in the wide feature payload.
long_schema = build_arrow_schema(SynapseConnectivityLong)
long_table = pa.Table.from_arrays(
    [
        pa.repeat(PROJECT_ID, synapse_count),
        pc.cast(synapse_source["id"], pa.string()),
        pc.cast(synapse_source["pre_pt_root_id"], pa.string()),
        pc.cast(synapse_source["post_pt_root_id"], pa.string()),
        pa.repeat(SYNAPSE_TABLE_ID, synapse_count),
    ],
    schema=long_schema,
)
long_table = attach_linkml_metadata(
    long_table,
    linkml_class="SynapseConnectivityLong",
)
write_deltalake(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["SynapseConnectivityLong"],
    long_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
del long_table
gc.collect()

# Root ids define connectivity and are excluded from the feature matrix. The
# label Feather stores synapse ids in its index, exposed here as the id column.
feature_columns = [
    column
    for column in synapse_source.column_names
    if column != "id" and not column.endswith("_root_id")
]
feature_table = synapse_source.select(["id"] + feature_columns).set_column(
    0,
    "id",
    pc.cast(synapse_source["id"], pa.string()),
)
label_table = feather.read_table(
    SYNAPSE_LABEL_PATH,
    memory_map=False,
).select(["id", "tag"])
label_count = label_table.num_rows
label_table = label_table.set_column(
    0,
    "id",
    pc.cast(label_table["id"], pa.string()),
).rename_columns(["id", "synaptictargetlabel"])

# Left join keeps every synapse; unlabeled targets remain null.
synapse_wide = feature_table.join(
    label_table,
    keys="id",
    join_type="left outer",
)
synapse_wide = synapse_wide.append_column(
    "project_id",
    pa.repeat(PROJECT_ID, synapse_count),
)
synapse_wide = synapse_wide.append_column(
    "synapse_table_id",
    pa.repeat(SYNAPSE_TABLE_ID, synapse_count),
)
synapse_feature_path = (
    OUTPUT_ROOT
    / WIDE_PAYLOAD_PATHS["synapse_features"]
    / SYNAPSE_FEATURE_MATRIX_ID
)
write_deltalake(
    synapse_feature_path,
    synapse_wide,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
synapse_feature_matrix = SynapseFeatureMatrix(
    id=SYNAPSE_FEATURE_MATRIX_ID,
    description="Per-synapse position, size and target label for V1DD 1196.",
    synapse_table_id=SYNAPSE_TABLE_ID,
    project_id=PROJECT_ID,
    parquet_path=f"file://{synapse_feature_path.resolve()}/",
    synapse_index_column="id",
)
print(write_models([synapse_feature_matrix], output_root=OUTPUT_ROOT))
print(f"synapses: {synapse_count:,} | labelled: {label_count:,}")
del synapse_source, feature_table, label_table, synapse_wide
gc.collect()

In [ ]:
# Read only columns needed for aggregation and restrict endpoints to the two
# registered cohorts before constructing cell-cell measurements.
connectivity_source = pl.from_arrow(
    feather.read_table(
        SYNAPSE_PATH,
        columns=["id", "pre_pt_root_id", "post_pt_root_id", "size"],
        memory_map=False,
    )
)
axon_ids = set(map(str, axon_proof_root_ids))
dendrite_ids = set(map(str, dendrite_proof_root_ids))
selected_synapses = (
    connectivity_source
    .filter(
        pl.col("pre_pt_root_id").is_in(set(map(int, axon_proof_root_ids)))
        & pl.col("post_pt_root_id").is_in(set(map(int, dendrite_proof_root_ids)))
    )
    .select(
        pl.col("id").cast(pl.String),
        pl.col("pre_pt_root_id").cast(pl.String).alias("presynaptic_cell"),
        pl.col("post_pt_root_id").cast(pl.String).alias("postsynaptic_cell"),
        pl.col("size"),
        pl.lit(PROJECT_ID).alias("project_id"),
        pl.lit(SYNAPSE_TABLE_ID).alias("synapse_table_id"),
    )
)
pair_count = selected_synapses.select(
    "presynaptic_cell",
    "postsynaptic_cell",
).unique().height

# The helper emits one SYNAPSE_COUNT and one SUM_ANATOMICAL_SIZE measurement
# for each connected cell pair, preserving the source synapse-table provenance.
connectivity = derive_cell_cell_connectivity(
    selected_synapses,
    project_id=PROJECT_ID,
    connectome_id=CONNECTOME_ID,
    modality=Modality.ELECTRON_MICROSCOPY,
    size_column="size",
    size_unit=Unit.ARBITRARY_UNIT,
)
connectivity_table = attach_linkml_metadata(
    cell_cell_connectivity_to_arrow(connectivity),
    linkml_class="CellCellConnectivityLong",
)
write_deltalake(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["CellCellConnectivityLong"],
    connectivity_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"connectome_id = '{CONNECTOME_ID}'"
    ),
    partition_by=["project_id", "connectome_id", "measurement_type"],
)
print(
    f"connectivity: {pair_count:,} pairs | "
    f"{connectivity_table.num_rows:,} measurements"
)
del connectivity_source, selected_synapses, connectivity, connectivity_table
gc.collect()

## soma features

In [ ]:
soma_features = pl.read_parquet(SOMA_FEATURES_PATH)

# These measurements are defined for soma ids. Keep registered proofread
# dendrites, cast the soma root to the CCM DataItem key, and enforce one row each.
soma_features_cc = (
    soma_features
    .with_columns(pl.col("soma_id").cast(pl.String).alias("id"))
    .filter(pl.col("id").is_in(list(dendrite_ids)))
    .unique(subset=["id"], keep="first")
)
assert set(soma_features_cc["id"].to_list()) <= dendrite_ids
feature_columns = [
    column for column in soma_features_cc.columns
    if column not in {"soma_id", "nucleus_id", "id"}
]
unit_by_feature = {
    "nucleus_volume_um": Unit.MICRONS_CUBED.value,
    "nucleus_area_um": Unit.MICRONS_SQUARE.value,
    "nuclear_area_to_volume_ratio": Unit.RATIO.value,
    "nuclear_folding_area_um": Unit.MICRONS_SQUARE.value,
    "fraction_nuclear_folding": Unit.RATIO.value,
    "nucleus_to_soma_ratio": Unit.RATIO.value,
    "soma_volume_um": Unit.MICRONS_CUBED.value,
    "soma_area_um": Unit.MICRONS_SQUARE.value,
    "soma_to_nucleus_center_dist": Unit.MICRONS_LENGTH.value,
    "soma_area_to_volume_ratio": Unit.RATIO.value,
    "soma_synapse_density_um": Unit.COUNT_PER_MICRONS_CUBED.value,
}
assert set(feature_columns) == set(unit_by_feature)
feature_definitions = [
    CellFeatureDefinition(
        id=column, description=column.replace("_", " "),
        unit=unit_by_feature[column], data_type="<f8",
        project_id=PROJECT_ID, feature_set_id=SOMA_FEATURE_SET_ID,
    )
    for column in feature_columns
]
soma_feature_set = CellFeatureSet(
    id=SOMA_FEATURE_SET_ID,
    description="V1DD soma morphology and nucleus-derived features.",
    feature_definition_ids=[definition.id for definition in feature_definitions],
    extraction_method="Loaded from soma_features.parquet.",
    project_id=PROJECT_ID,
)
print(write_models(feature_definitions, output_root=OUTPUT_ROOT))
print(write_models([soma_feature_set], output_root=OUTPUT_ROOT))

# Store measurements in a typed wide table and register its location separately.
soma_wide = (
    soma_features_cc.select(["id"] + feature_columns)
    .with_columns([pl.col(column).cast(pl.Float64) for column in feature_columns])
    .to_pandas()
)
soma_wide["project_id"] = PROJECT_ID
soma_wide["feature_set_id"] = SOMA_FEATURE_SET_ID
soma_schema = build_cell_feature_matrix_schema(
    soma_feature_set, feature_definitions, cell_index_column="id"
)
soma_table = pa.Table.from_pandas(
    soma_wide, schema=soma_schema, preserve_index=False
)
soma_feature_path = (
    OUTPUT_ROOT / WIDE_PAYLOAD_PATHS["cell_features"] / SOMA_FEATURE_SET_ID
)
write_deltalake(
    soma_feature_path, soma_table, mode="overwrite",
    predicate=f"project_id = '{PROJECT_ID}'",
    partition_by=["project_id", "feature_set_id"],
)
soma_matrix = CellFeatureMatrix(
    id=f"{PROJECT_ID}_{SOMA_FEATURE_SET_ID}",
    feature_set_id=SOMA_FEATURE_SET_ID,
    parquet_path=f"file://{soma_feature_path.resolve()}/",
    cell_index_column="id", project_id=PROJECT_ID,
)
print(write_models([soma_matrix], output_root=OUTPUT_ROOT))
print(f"soma feature rows: {soma_table.num_rows:,}")

## cell type colors

In [ ]:
rtf_text = COLORS_PATH.read_text(encoding="cp1252")
palette = dict(re.findall(r'"([^"]+)"\s*:\s*"(#[0-9A-Fa-f]{6})"', rtf_text))
assert len(palette) == 14
cluster_frame = DeltaTable(str(OUTPUT_ROOT / MODEL_TABLE_PATHS["Cluster"])).to_pandas()
cluster_frame = cluster_frame.loc[cluster_frame["hierarchy_id"] == HIERARCHY_ID].copy()
# The palette is defined at subclass level. Leaves inherit their parent color,
# with aliases bridging source labels to the palette vocabulary.
source_to_palette = {"L5": "L5IT", "BipolarFamily": "Bipolarfamily"}
used_palette_keys = set()

def palette_key(source_name):
    return source_to_palette.get(source_name, source_name)

def resolve_color(row):
    if row["hierarchy_category"] == SUBCLASS_CATEGORY:
        key = palette_key(row["id"])
    elif row["hierarchy_category"] == CLUSTER_CATEGORY:
        source_cluster = row["id"].split(":", maxsplit=1)[1]
        key = "L6b" if source_cluster == "L6b" else palette_key(row["parent"])
    else:
        return None
    color = palette.get(key)
    if color is not None:
        used_palette_keys.add(key)
    return color

cluster_frame["hex_color"] = cluster_frame.apply(resolve_color, axis=1)
expected_uncolored = set(
    cluster_frame.loc[
        cluster_frame["hierarchy_category"].isna()
        | cluster_frame["hierarchy_category"].eq(CLASS_CATEGORY),
        "id",
    ]
)
assert set(cluster_frame.loc[cluster_frame["hex_color"].isna(), "id"]) == expected_uncolored
assert used_palette_keys == set(palette)

def optional_value(value):
    return None if value is None or pd.isna(value) else value

colored_clusters = [
    Cluster(
        id=str(row.id), hierarchy_id=HIERARCHY_ID,
        parent=optional_value(row.parent),
        children=list(row.children) if row.children is not None and len(row.children) else None,
        level=None if pd.isna(row.level) else int(row.level),
        hex_color=optional_value(row.hex_color),
        hierarchy_category=optional_value(row.hierarchy_category),
    )
    for row in cluster_frame.itertuples()
]
print(write_models(colored_clusters, output_root=OUTPUT_ROOT))
print(f"colored clusters: {sum(row.hex_color is not None for row in colored_clusters)} / {len(colored_clusters)}")

## embeddings

In [ ]:
embedding_columns = {
    QC_EMBEDDING_ID: ("qc_nonnull_embed0", "qc_nonnull_embed1"),
    AD_REF_EMBEDDING_ID: ("ad_ref_embed0", "ad_ref_embed1"),
}
embedding_spaces = [
    EmbeddingSpace(
        id=QC_EMBEDDING_ID,
        name="V1DD r84 QC-nonnull UMAP",
        description="QC-nonnull UMAP coordinates from the r84 cell-type parquet.",
        embedding_method=EmbeddingMethod.UMAP,
        parameters_json=json.dumps(
            {"source_columns": embedding_columns[QC_EMBEDDING_ID]}
        ),
        input_features_description=(
            "QC-nonnull axon/dendrite features supplied in the r84 parquet."
        ),
        project_id=PROJECT_ID,
    ),
    EmbeddingSpace(
        id=AD_REF_EMBEDDING_ID,
        name="V1DD r84 axon-dendrite reference UMAP",
        description=(
            "Axon-dendrite reference UMAP coordinates from the r84 cell-type parquet."
        ),
        embedding_method=EmbeddingMethod.UMAP,
        parameters_json=json.dumps(
            {"source_columns": embedding_columns[AD_REF_EMBEDDING_ID]}
        ),
        input_features_description=(
            "Reduced axon-dendrite reference features supplied in the r84 parquet."
        ),
        project_id=PROJECT_ID,
    ),
]
print(write_models(embedding_spaces, output_root=OUTPUT_ROOT))

# Embedding rows use nucleus ids, so translate them to the canonical CCM
# DataItem ids before creating locations.
dataitem_lookup = pl.DataFrame({
    "nucleus_id": soma_ct_unique_df["id"].astype(str),
    "dataitem_id": soma_ct_unique_df["pt_root_id"].astype(str),
})
embedding_source = (
    pl.read_parquet(R84_PATH)
    .with_columns(pl.col("cell_id").cast(pl.String).alias("nucleus_id"))
    .join(dataitem_lookup, on="nucleus_id", how="inner")
    .unique(subset=["dataitem_id"], keep="first")
)
locations_by_space = {}
for embedding_space, (x_column, y_column) in embedding_columns.items():
    # Missing coordinates are space-specific; retain every finite point per space.
    coordinates = embedding_source.filter(
        pl.col(x_column).is_finite() & pl.col(y_column).is_finite()
    )
    locations = [
        EmbeddingLocation(
            dataitem_id=row["dataitem_id"],
            embedding_space=embedding_space,
            x=float(row[x_column]),
            y=float(row[y_column]),
            project_id=PROJECT_ID,
        )
        for row in coordinates.iter_rows(named=True)
    ]
    locations_by_space[embedding_space] = locations
    print(write_models(locations, output_root=OUTPUT_ROOT))
    print(f"{embedding_space}: {len(locations):,}")

## verify

In [ ]:
# Read each CCM model table within this project's scope so unrelated projects
# sharing the output root do not affect the checks.
stored_datasets = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataSet"]).filter(
    pl.col("project_id") == PROJECT_ID
)
stored_items = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItem"]).filter(
    pl.col("project_id") == PROJECT_ID
)
stored_associations = pl.read_delta(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["DataItemDataSetAssociation"]
).filter(pl.col("project_id") == PROJECT_ID)
stored_locations = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["SpatialLocation"]).filter(
    pl.col("project_id") == PROJECT_ID
)
stored_clusters = pl.read_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["Cluster"]).filter(
    pl.col("hierarchy_id") == HIERARCHY_ID
)
stored_memberships = pl.read_delta(
    OUTPUT_ROOT / MODEL_TABLE_PATHS["ClusterMembership"]
).filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("hierarchy_id") == HIERARCHY_ID)
)

# Confirm expected row counts, unique spatial keys, and valid references between
# datasets, DataItems, hierarchy nodes, and memberships.
assert set(stored_datasets["id"]) == {
    DATASET_EM,
    DATASET_PROOFREAD_AXON,
    DATASET_PROOFREAD_DEND,
}
assert stored_items.height == len(soma_ct_unique_df)
assert set(stored_items["id"]) == registered_ids
assert stored_locations.height == 2 * len(soma_ct_unique_df)
assert stored_locations.select(
    "dataitem_id",
    "reference_space",
    "location_type",
).unique().height == stored_locations.height
association_counts = dict(
    stored_associations.group_by("dataset_id").len().iter_rows()
)
assert association_counts == {
    DATASET_EM: len(unique_root_ids),
    DATASET_PROOFREAD_AXON: len(axon_proof_root_ids),
    DATASET_PROOFREAD_DEND: len(dendrite_proof_root_ids),
}
assert stored_clusters.height == len(clusters) == 149
assert stored_clusters.filter(pl.col("hex_color").is_not_null()).height == 146
assert stored_memberships.height == 230918
assert set(stored_memberships["item"]) <= registered_ids
assert set(stored_memberships["cluster"]) <= set(stored_clusters["id"])

# Wide feature payloads should contain one stored row per retained DataItem in
# the population used to build each matrix.
stored_volume = pl.read_delta(volume_path).filter(pl.col("project_id") == PROJECT_ID)
stored_soma_features = pl.read_delta(soma_feature_path).filter(
    pl.col("project_id") == PROJECT_ID
)
assert stored_volume.height == len(soma_ct_unique_df)
assert stored_soma_features.height == soma_features_cc.height

# Stream counts for large Delta tables so verification does not materialize
# millions of rows in memory.
stored_synapse_count = (
    pl.scan_delta(OUTPUT_ROOT / MODEL_TABLE_PATHS["SynapseConnectivityLong"])
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)
stored_synapse_feature_count = (
    pl.scan_delta(synapse_feature_path)
    .filter(
        (pl.col("project_id") == PROJECT_ID)
        & (pl.col("synapse_table_id") == SYNAPSE_TABLE_ID)
    )
    .select(pl.len())
    .collect(engine="streaming")
    .item()
)

# Each connected pair should have the two derived measurement types, while the
# long synapse table and its feature matrix remain row-aligned.
stored_connectivity = read_cell_cell_connectivity(
    PROJECT_ID,
    CONNECTOME_ID,
    synapse_table_id=SYNAPSE_TABLE_ID,
    output_root=OUTPUT_ROOT,
)
assert stored_synapse_count == synapse_count
assert stored_synapse_feature_count == synapse_count
assert stored_connectivity.height == pair_count * 2
assert stored_connectivity["measurement_type"].n_unique() == 2

# Every embedding location must belong to a registered DataItem and match the
# number of finite source coordinates written for that embedding space.
for embedding_space, locations in locations_by_space.items():
    stored = read_embedding_locations(
        PROJECT_ID,
        embedding_space_ids=embedding_space,
        output_root=OUTPUT_ROOT,
    )
    assert stored.height == len(locations)
    assert set(stored["dataitem_id"]) <= registered_ids
print(
    "V1DD 1196 datasets, spatial data, features, taxonomy, "
    "synapses, connectivity, colors, and embeddings verified."
)